In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

In [ ]:
import io
import os

from utils.funciones_minio import crear_cliente_minio, bajar_minio
from utils.config import PATH_PRIMARIOS_LIMPIO

OBJ_VIVIENDAS_VENTA = "viviendas_venta.parquet"
OBJ_VIVIENDAS_ALQUILER = "viviendas_alquiler.parquet"


In [ ]:
client = crear_cliente_minio()

In [ ]:
df_venta = bajar_minio(client, PATH_PRIMARIOS_LIMPIO, OBJ_VIVIENDAS_VENTA)
df_alquiler = bajar_minio(client, PATH_PRIMARIOS_LIMPIO, OBJ_VIVIENDAS_ALQUILER)

In [ ]:
df_venta["tipo"] = "venta"
df_alquiler["tipo"] = "alquiler"

df = pd.concat([df_venta, df_alquiler])

In [ ]:
df.columns

In [ ]:
def agrupar_tipo(x):
    if x == "Particular":
        return "Particular"
    elif x in ["Agente Pro", "Profesional"]:
        return "Intermediario"
    elif x == "Promotora":
        return "Promotora"

df["grupo"] = df["Anuncia"].apply(agrupar_tipo)

df["grupo"].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [ ]:
"""import wandb

wandb.init(
    project="clasificacion-anuncios-vivienda",
    config={
        "max_features": 20000,
        "ngram_range": (1,2),
        "min_df": 3,
        "max_df": 0.9,
        "C": 1.0
    }
)
config = wandb.config"""

In [ ]:
X = df["Descripcion"]
y = df["grupo"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("vectorizer", CountVectorizer(
        max_features=20000,
        ngram_range=(1,2),
        min_df=3,
        max_df=0.9
    )),
    
    ("classifier", LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
from imblearn.over_sampling import RandomOverSampler

vectorizer = CountVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9
)

classifier = LogisticRegression(
    C=1.0,
    penalty="l2",
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced"
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

ros = RandomOverSampler()
X_train_res, y_train_res = ros.fit_resample(X_train_vec, y_train)

classifier.fit(X_train_res, y_train_res)

y_pred = classifier.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))